# Phase 2: Learn Retrieval from First Principles

## Step 3: Brute-Force Embedding Retrieval

### Learning

- Embeddings
- Vectors
- Semantic similarity
- Cosine similarity
- Cosine distance
- Retrieval-Augmented Generation (RAG)
- Retrieval and generation as separate stages
- Top-k retrieval

---

## Key Takeaways

- Embeddings convert text into vectors.
- Similar meanings produce similar vectors.
- Cosine similarity measures how close two vectors are.
- Retrieval finds relevant information before generation.
- RAG improves answers by using retrieved context.
- Top-k retrieval returns the most relevant chunks.

## 1. Environment Setup & Imports

In [16]:
import sys
from pathlib import Path

# Allow imports from project root (one level up from notebooks/)
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import gradio as gr
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME,EMBEDDING_MODEL

client = OpenAI(api_key=OPENAI_API_KEY)

print("API key loaded:", bool(OPENAI_API_KEY))
print("Model:", MODEL_NAME)

API key loaded: True
Model: gpt-4.1-mini


In [ ]:
## helper method to reload specific file
import importlib
import config
importlib.reload(config)

<module 'config' from '/Users/hirakhan/Developer/AI-ML/rag-chatbot/config.py'>

## 2. Load Document

Load the reference document from `data/profile1.txt` into memory.

In [8]:
document_path = PROJECT_ROOT / "data" / "profile.txt"

try:
    document_text = document_path.read_text(encoding="utf-8")
    print("Document loaded successfully.")
    print(f"Characters: {len(document_text)}")
except FileNotFoundError:
    document_text = ""
    print(f"Document not found at {document_path}")

Document loaded successfully.
Characters: 8617


###  -> Splitting the document in paragraphs and assigning the unique IDS

In [22]:
paragraphs =[
    paragraph.strip()
    for paragraph in document_text.split("\n\n")
    if paragraph.strip()
]

documents= [
    {
        "id":i,
        "text":para
    }
    for i, para in enumerate(paragraphs)
]
print(f"Total paragraphs!!!: {len(documents)}")
documents[0]


Total paragraphs!!!: 22


{'id': 0,
 'text': 'Fictional Biography of John Doe\nJohn Doe: A Life of Curiosity, Innovation, and Service'}

### -> Embedding the document

In [ ]:
# -------------- SAVING THE EMBEDDED DOCS IN THE SAME OBJECT
# for doc in documents:
#     response = client.embeddings.create(
#         model=EMBEDDING_MODEL,
#         input=doc["text"]
#     )
    #   LOOP ----
#     doc["embedding"] = response.data[0].embedding

#     print(documents[0].keys())
# # documents[0]

# print(len(documents[0]["embedding"]))

In [24]:
embedded_documents = []

for doc in documents:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=doc["text"]
    )

    embedded_documents.append({
        "id": doc["id"],
        "text": doc["text"],
        "embedding": response.data[0].embedding
    })

In [25]:
print(documents[0].keys())
print(embedded_documents[0].keys())

dict_keys(['id', 'text'])
dict_keys(['id', 'text', 'embedding'])


In [30]:
query= "who is john doe?"

query_embedding= client.embeddings.create(
    model= EMBEDDING_MODEL,
    input=query
)

query_vector= query_embedding.data[0].embedding

# print(len(query_vector))

document_dimension = len(embedded_documents[0]["embedding"])
query_dimension = len(query_vector)

print("Document embedding:", document_dimension)
print("Query embedding:", query_dimension)

print(document_dimension == query_dimension)

Document embedding: 1536
Query embedding: 1536
True


In [35]:
import numpy as np 

def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)

    return np.dot(a, b) / (
        np.linalg.norm(a) * np.linalg.norm(b)
    )

In [37]:
result_score=[]

for doc in embedded_documents:
    score = cosine_similarity(
        query_vector,
        doc["embedding"]
    )

    result_score.append({
        "id":doc["id"],
        "score":score,
        "text":doc["text"]
    })

In [38]:
results = sorted(
    result_score,
    key=lambda x: x["score"],
    reverse=True
)

In [ ]:
# DEBUGGING ---------

print(f"{'ID':<8} {'Score':<10} Preview")
print("-" * 70)

for result in results:

    preview = result["text"][:70].replace("\n", " ")

    print(
        f"{result['id']:<8}"
        f"{result['score']:.4f}    "
        f"{preview}"
    )

ID       Score      Preview
----------------------------------------------------------------------
1       0.6740    John Alexander Doe was born on March 18, 1987, in the quiet town of Br
0       0.5974    Fictional Biography of John Doe John Doe: A Life of Curiosity, Innovat
2       0.5592    John was the eldest of three children born to Michael and Sarah Doe. H
15      0.4294    Friends often described John as patient, dependable, and endlessly cur
14      0.3822    Outside of his professional life, John maintained a wide variety of in
3       0.3811    During his childhood, John excelled academically but was equally inter
11      0.3705    Throughout his career, John believed that technology should always ser
9       0.3612    Over the next several years, John gained expertise in full-stack softw
21      0.3575    John's fictional story illustrates how curiosity, continuous learning,
5       0.3523    After graduating with honors, John enrolled at the fictional North Val
19      0.3

In [40]:
best_match = results[0]

print("Document ID:")
print(best_match["id"])

print("\nSimilarity:")
print(best_match["score"])

print("\nRetrieved paragraph:\n")
print(best_match["text"])

Document ID:
1

Similarity:
0.6739986295200107

Retrieved paragraph:

John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.


In [42]:
# Measure retrieval time
import time

start = time.perf_counter()

results = []

for doc in embedded_documents:

    score = cosine_similarity(
        query_vector,
        doc["embedding"]
    )

    results.append({
        "id": doc["id"],
        "score": score,
        "text": doc["text"]
    })

results.sort(
    key=lambda x: x["score"],
    reverse=True
)

best_match = results[0]

retrieval_time = time.perf_counter() - start

print(f"Retrieval Time: {retrieval_time:.6f} seconds")

Retrieval Time: 0.003354 seconds


In [43]:
print("=" * 60)

print("Retrieved Context")
print("-" * 60)
print(best_match["text"])

print("\nSimilarity Score:")
print(best_match["score"])

print("\nDocument ID:")
print(best_match["id"])

print(f"\nRetrieval Time: {retrieval_time:.6f} seconds")

print("=" * 60)

Retrieved Context
------------------------------------------------------------
John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.

Similarity Score:
0.6739986295200107

Document ID:
1

Retrieval Time: 0.003354 seconds


In [44]:
messages = [
    {
        "role": "system",
        "content": f"""
You answer questions using ONLY the retrieved context below.

If the answer is not present, say:
'I could not find that information in the document.'

Retrieved Context:

{best_match["text"]}
"""
    },
    {
        "role": "user",
        "content": query
    }
]

response = client.chat.completions.create(
    model="gpt-4.1-mini",
    messages=messages
)

print(response.choices[0].message.content)

John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked, often dismantling old radios and spending hours reading books from the local library, which helped develop his passion for learning.


In [45]:
test_queries = [
    # Same wording
    "Where did John Doe study?",

    # Similar meaning
    "What university did John Doe attend?",

    # Multiple paragraphs
    "Summarize John Doe's education and career.",

    # Exact names
    "Emily Carter",

    # Product code / IDs
    "AUR-204",

    # Very short
    "Education",

    # Ambiguous
    "Where?"
]

In [46]:
for query in test_queries:

    print("=" * 80)
    print("Query:")
    print(query)

    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=query
    )

    query_embedding = response.data[0].embedding

    results = []

    for doc in embedded_documents:

        score = cosine_similarity(
            query_embedding,
            doc["embedding"]
        )

        results.append({
            "id": doc["id"],
            "score": score,
            "text": doc["text"]
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    best = results[0]

    print("\nBest Match:")
    print(best["id"])
    print(best["score"])
    print(best["text"][:150])

Query:
Where did John Doe study?

Best Match:
1
0.6060671463423275
John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neig
Query:
What university did John Doe attend?

Best Match:
1
0.5889744748841956
John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neig
Query:
Summarize John Doe's education and career.

Best Match:
0
0.5621375456930269
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service
Query:
Emily Carter

Best Match:
14
0.21659225089442755
Outside of his professional life, John maintained a wide variety of interests. He enjoyed hiking, landscape photography, reading historical biographie
Query:
AUR-204

Best Match:
17
0.17147078967955437
As artificial intelligence continued to evolve throughout the 2020s, John expanded his work into generative AI, retrieval-au

In [32]:
# CHECKING THE TOKENS 
import tiktoken
encoding = tiktoken.encoding_for_model(EMBEDDING_MODEL)

tokens = len(encoding.encode(embedded_documents[0]["text"]))

tokens

21

In [47]:
def retrieve(query_embedding, documents, top_k=1):

    results = []

    for doc in documents:

        score = cosine_similarity(
            query_embedding,
            doc["embedding"]
        )

        results.append({
            "id": doc["id"],
            "score": score,
            "text": doc["text"]
        })

    results.sort(
        key=lambda x: x["score"],
        reverse=True
    )

    return results[:top_k]

In [48]:
top_results = retrieve(
    query_vector,
    embedded_documents,
    top_k=1
)

for result in top_results:

    print(result["id"])
    print(result["score"])
    print(result["text"])

1
0.6739986295200107
John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.


In [51]:
top_results = retrieve(
    query_vector,
    embedded_documents,
    top_k=3
)

for i, result in enumerate(top_results, start=1):

    print("=" * 70)
    print(f"Rank {i}")
    print(result["id"])
    print(result["score"])
    print(result["text"])

Rank 1
1
0.6739986295200107
John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.
Rank 2
0
0.597374887645467
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service
Rank 3
2
0.5591734690899917
John was the eldest of three children born to Michael and Sarah Doe. His father worked as a mechanical engineer, while his mother was a high school English teacher. Growing up in a household that valued both analytical thinking and creativity, John learned the importance of balancing logic with imagination. Family evenings often consisted of discussions about science, li

In [52]:
top_results = retrieve(
    query_vector,
    embedded_documents,
    top_k=5
)

for i, result in enumerate(top_results, start=1):

    print("=" * 70)
    print(f"Rank {i}")
    print(result["id"])
    print(result["score"])
    print(result["text"])

Rank 1
1
0.6739986295200107
John Alexander Doe was born on March 18, 1987, in the quiet town of Brookfield, a close-knit community known for its tree-lined streets, friendly neighborhoods, and strong sense of community. From an early age, John displayed an unusual curiosity about how things worked. Whether dismantling old radios in his parents' garage or spending hours reading books from the local library, he developed a passion for learning that would shape the rest of his life.
Rank 2
0
0.597374887645467
Fictional Biography of John Doe
John Doe: A Life of Curiosity, Innovation, and Service
Rank 3
2
0.5591734690899917
John was the eldest of three children born to Michael and Sarah Doe. His father worked as a mechanical engineer, while his mother was a high school English teacher. Growing up in a household that valued both analytical thinking and creativity, John learned the importance of balancing logic with imagination. Family evenings often consisted of discussions about science, li

In [53]:
retrieved_context = "\n\n".join(
    result["text"]
    for result in top_results
)

In [54]:
messages = [
    {
        "role": "system",
        "content": f"""
You answer questions ONLY using the retrieved context.

Retrieved Context:

{retrieved_context}
"""
    },
    {
        "role": "user",
        "content": query
    }
]

## 3. Standalone RAG Chatbot (Combined Cell for Gradio ChatInterface)

This single cell combines document loading, chunking, embedding generation, brute-force cosine similarity retrieval, and the Gradio ChatInterface.

In [55]:
import sys
from pathlib import Path
import numpy as np
import gradio as gr
from openai import OpenAI
from config import OPENAI_API_KEY, MODEL_NAME, EMBEDDING_MODEL

# 1. Initialize client & resolve paths
PROJECT_ROOT = Path("..").resolve()
client = OpenAI(api_key=OPENAI_API_KEY)

# 2. Load document & split into paragraphs
document_path = PROJECT_ROOT / "data" / "profile.txt"
document_text = document_path.read_text(encoding="utf-8")
paragraphs = [p.strip() for p in document_text.split("\n\n") if p.strip()]
documents = [{"id": i, "text": p} for i, p in enumerate(paragraphs)]

# 3. Compute embeddings for all document paragraphs
embedded_documents = []
for doc in documents:
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=doc["text"]
    )
    embedded_documents.append({
        "id": doc["id"],
        "text": doc["text"],
        "embedding": response.data[0].embedding
    })

# 4. Helper functions for Cosine Similarity & Retrieval
def cosine_similarity(a, b):
    a = np.array(a)
    b = np.array(b)
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query_vector, docs, top_k=3):
    scores = []
    for doc in docs:
        score = cosine_similarity(query_vector, doc["embedding"])
        scores.append({"id": doc["id"], "score": score, "text": doc["text"]})
    scores.sort(key=lambda x: x["score"], reverse=True)
    return scores[:top_k]

# 5. RAG Chatbot function for Gradio ChatInterface
def rag_chatbot(message, history):
    # Embed the user query
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=message
    )
    query_vector = response.data[0].embedding
    
    # Retrieve top 3 relevant paragraphs
    top_docs = retrieve(query_vector, embedded_documents, top_k=3)
    retrieved_context = "\n\n".join([d["text"] for d in top_docs])
    
    # Build RAG system prompt
    system_prompt = f"""You answer questions using ONLY the retrieved context below.

If the answer is not present in the context, say:
'I could not find that information in the document.'

Retrieved Context:

{retrieved_context}"""

    messages = [{"role": "system", "content": system_prompt}]
    for user_msg, bot_msg in history:
        messages.append({"role": "user", "content": user_msg})
        messages.append({"role": "assistant", "content": bot_msg})
    messages.append({"role": "user", "content": message})
    
    chat_response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages
    )
    return chat_response.choices[0].message.content

# 6. Launch Gradio ChatInterface
demo = gr.ChatInterface(
    fn=rag_chatbot,
    title="Step 3: RAG Chatbot (Brute-Force Embedding Retrieval)",
    description="Ask questions about John Doe's profile. Uses vector embeddings & cosine similarity for top-k paragraph retrieval."
)

demo.launch(prevent_thread_lock=True)

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


In [56]:
demo.close()

Closing server running on port: 7862
